# Reconocimiento de rostros · Comparación completa (Práctica 2, 3.er parcial)

¿El modelo aprende mejor con una CNN propia o reutilizando redes ya entrenadas? Se comparan, con **las mismas caras y
la misma división**, cuatro formas de construir el reconocedor de las 29 personas del grupo:

| | Enfoque | Qué se entrena |
|---|---|---|
| **A** | Extracción de características: **ArcFace** (antelopev2) congelado + Ridge / SVM / KNN | solo el clasificador (el enfoque de Kenya) |
| **B1** | *Fine-tuning* de **InceptionResnetV1 preentrenada en caras (VGGFace2)** | capa de salida nueva + últimos bloques |
| **B2** | *Fine-tuning* de **ResNet-18 preentrenada en ImageNet** (objetos, no caras) | capa de salida nueva + toda la red |
| **C** | **CNN desde cero** (pesos aleatorios) | toda la red |

Además se repite la prueba con las caras de *test* **modificadas** (lentes oscuros, gorra, cubrebocas o bufanda, borrosa,
poca luz y baja resolución), porque el día de la entrega el profesor fotografiará a alguien **disfrazado**.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo → GPU (T4)`. En Drive, si `escom_caras` está en
*Compartido conmigo*: clic derecho → **Organizar → Agregar acceso directo → Mi unidad**. Tiempo aproximado en T4: 1.5–2 h.

## 0. Datos

Se usan las caras ya recortadas de `escom_caras` (las mismas del dataset compartido de los tres equipos, recortadas con
MediaPipe) y su `caras_manifest.csv` (persona, set, ruta). El notebook copia las ~29 mil caras a Colab para no leer del
Drive en cada época. Los resultados se guardan en `escom_caras/resultados_comparacion_<division>/` y al final se descarga
un zip sin los pesos.

Las fotos son datos biométricos de compañeros: los resultados se quedan en el Drive del grupo.

## 1. Configuración

In [ ]:
from pathlib import Path

# 'por_set'  = sesiones completas en test (ropa, fondo y luz que el modelo no vio): la prueba honesta, como el examen.
# 'aleatoria' = 20 % de fotos al azar (comparable con la corrida de Keras, pero mezcla fotos de la misma sesión).
DIVISION = 'por_set'   #@param ['por_set', 'aleatoria']
CARPETA_DRIVE = ''     # vacío = se busca sola la carpeta que tenga caras_manifest.csv dentro de MyDrive

TRABAJO     = Path('/content/reco')   # copia local (más rápida que leer del Drive)
SEMILLA     = 42

# Qué modelos entrenar (quita los que no quieras correr; B1 es el más lento)
MODELOS = ['C_cnn_cero', 'B1_facenet_ft', 'B2_resnet18_ft']
CORRER_LINEA_BASE_A = True     # ArcFace + Ridge / SVM / KNN (el enfoque de Kenya)

LOTE = 64

## 2. Entorno

In [ ]:
!nvidia-smi -L || echo 'SIN GPU: Entorno de ejecucion -> Cambiar tipo -> T4'
# facenet-pytorch fija versiones viejas de torch; --no-deps evita que desinstale el torch de Colab.
!pip install -q facenet-pytorch --no-deps

In [ ]:
import json, random, time, copy, os, zipfile, shutil
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import v2

from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
                             classification_report, confusion_matrix)

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| dispositivo:', DISPOSITIVO)
if DISPOSITIVO == 'cpu':
    print('AVISO: sin GPU el entrenamiento tarda muchisimo. Cambia el tipo de entorno a T4.')


def fijar_semilla(s=SEMILLA):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


fijar_semilla()

## 3. Cargar datos

In [ ]:
from google.colab import drive
from concurrent.futures import ThreadPoolExecutor
try:
    drive.mount('/content/drive')
except ValueError:
    # Suele pasar si se cerró la ventana de permisos o faltó marcar alguna casilla; se reintenta con más tiempo.
    print('No se pudo conectar Drive. Se reintenta: en la ventana elige la cuenta que tiene escom_caras')
    print('y marca TODAS las casillas de permisos.')
    drive.mount('/content/drive', force_remount=True, timeout_ms=300000)

if CARPETA_DRIVE:
    RAIZ = Path(CARPETA_DRIVE)
else:
    directo = Path('/content/drive/MyDrive/escom_caras/caras_manifest.csv')
    candidatos = [directo] if directo.exists() else \
        sorted(Path('/content/drive/MyDrive').glob('**/caras_manifest.csv'), key=lambda p: len(p.parts))
    assert candidatos, 'No encontré caras_manifest.csv: agrega el acceso directo a escom_caras en Mi unidad o llena CARPETA_DRIVE'
    RAIZ = candidatos[0].parent
SALIDA = str(RAIZ / f'resultados_comparacion_{DIVISION}')
Path(SALIDA).mkdir(parents=True, exist_ok=True)
print('Datos:', RAIZ, '| salidas:', SALIDA)

manifest = pd.read_csv(RAIZ / 'caras_manifest.csv')
ORIGEN, LOCAL = RAIZ / 'caras', TRABAJO / 'caras'

def copiar(ruta):
    destino = LOCAL / ruta
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(ORIGEN / ruta, destino)

t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(copiar, manifest['ruta']), 1):
        if i % 5000 == 0:
            print(f'{i}/{len(manifest)} ({time.time() - t0:.0f} s)')
print(f'{len(manifest)} caras copiadas en {time.time() - t0:.0f} s')

In [ ]:
# --- División -----------------------------------------------------------------------------------------------
# por_set: por persona se eligen sets completos cuyo tamaño sume ~20 % (test) y ~20 % (val); misma regla que
# src/dividir_sets.py del pipeline de Kenya. aleatoria: estratificada por persona.
from sklearn.model_selection import train_test_split
FRAC_VAL, FRAC_TEST, MIN_SETS_PARA_VALIDACION = 0.20, 0.20, 4


def mejor_combinacion(tam, frac, total, dejar):
    mejor, mejor_err = (), np.inf
    for k in range(1, len(tam) - dejar + 1):
        for c in combinations(tam.index, k):
            err = abs(tam[list(c)].sum() - frac * total)
            if err < mejor_err:
                mejor, mejor_err = c, err
    return mejor


def asignar_split(datos):
    split = pd.Series('train', index=datos.index)
    for persona, g in datos.groupby('persona'):
        total, tam = len(g), g.groupby('set').size()
        test = mejor_combinacion(tam, FRAC_TEST, total, dejar=2)
        split[(datos['persona'] == persona) & datos['set'].isin(test)] = 'test'
        resto = tam.drop(list(test))
        if len(resto) >= MIN_SETS_PARA_VALIDACION:
            val = mejor_combinacion(resto, FRAC_VAL, total, dejar=2)
            split[(datos['persona'] == persona) & datos['set'].isin(val)] = 'val'
    return split


datos = manifest[['persona', 'set', 'ruta']].copy()
if DIVISION == 'por_set':
    datos['split'] = asignar_split(datos)
    assert datos.groupby(['persona', 'set'])['split'].nunique().max() == 1, 'Hay sets repartidos entre particiones'
else:
    resto, test = train_test_split(datos, test_size=FRAC_TEST, stratify=datos['persona'], random_state=SEMILLA)
    _, val = train_test_split(resto, test_size=FRAC_VAL / (1 - FRAC_TEST), stratify=resto['persona'], random_state=SEMILLA)
    datos['split'] = 'train'
    datos.loc[val.index, 'split'] = 'val'
    datos.loc[test.index, 'split'] = 'test'
print('División:', DIVISION)

datos['png'] = [LOCAL / r for r in datos['ruta']]
CLASES = sorted(datos['persona'].unique())
A_INDICE = {p: i for i, p in enumerate(CLASES)}
datos['y'] = datos['persona'].map(A_INDICE)
N_CLASES = len(CLASES)

tabla = pd.crosstab(datos['persona'], datos['split']).reindex(columns=['train', 'val', 'test'], fill_value=0)
print(f'{N_CLASES} personas | {len(datos)} caras |', datos['split'].value_counts().to_dict())
tabla

In [ ]:
# Todas las caras a memoria (≈ 29 mil x 112x112x3 ≈ 1.1 GB): asi cada epoca no lee del disco.
def cargar(rutas, lado=112):
    X = np.empty((len(rutas), lado, lado, 3), np.uint8)
    for i, r in enumerate(rutas):
        img = Image.open(r).convert('RGB')
        if img.size != (lado, lado):
            img = img.resize((lado, lado), Image.BILINEAR)
        X[i] = np.asarray(img)
    return X


t0 = time.time()
X_TODO = cargar(datos['png'])
Y_TODO = datos['y'].to_numpy()
IDX = {s: np.flatnonzero(datos['split'].to_numpy() == s) for s in ['train', 'val', 'test']}
print(f'Cargadas en {time.time() - t0:.0f} s |', {k: len(v) for k, v in IDX.items()})

# Vistazo: una cara por persona
fig, ejes = plt.subplots(3, 10, figsize=(15, 5))
for ax, p in zip(ejes.flat, CLASES + [None] * 30):
    ax.axis('off')
    if p is not None:
        ax.imshow(X_TODO[np.flatnonzero(Y_TODO == A_INDICE[p])[0]]); ax.set_title(p[:14], fontsize=7)
plt.tight_layout(); plt.show()

## 4. Aumento de datos y *dataloaders*

Mismo aumento para todos los modelos (solo cambia el tamaño de entrada y la normalización que espera cada red):
recorte y escala aleatorios, espejo horizontal, giro de ±10°, cambios de brillo/contraste/color y a veces gris.
Validación y prueba **sin** aumento.

In [ ]:
NORMAS = {  # tamano de entrada y normalizacion que espera cada red
    'C_cnn_cero':     dict(lado=112, media=[0.5, 0.5, 0.5], desv=[0.5, 0.5, 0.5]),
    'B1_facenet_ft':  dict(lado=160, media=[0.5, 0.5, 0.5], desv=[0.5, 0.5, 0.5]),   # facenet: (x-127.5)/128
    'B2_resnet18_ft': dict(lado=112, media=[0.485, 0.456, 0.406], desv=[0.229, 0.224, 0.225]),  # ImageNet
}


def transformaciones(nombre, entrenar):
    n = NORMAS[nombre]
    final = [v2.ToDtype(torch.float32, scale=True), v2.Normalize(n['media'], n['desv'])]
    if not entrenar:
        return v2.Compose([v2.Resize(n['lado'], antialias=True)] + final)
    return v2.Compose([
        v2.RandomResizedCrop(n['lado'], scale=(0.80, 1.0), ratio=(0.9, 1.1), antialias=True),
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(10),
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        v2.RandomGrayscale(p=0.1),
    ] + final)


class Caras(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = idx, tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        img = torch.from_numpy(X_TODO[j]).permute(2, 0, 1)   # (3, H, W) uint8
        return self.tf(img), int(Y_TODO[j])


def cargadores(nombre):
    g = torch.Generator().manual_seed(SEMILLA)
    kw = dict(batch_size=LOTE, num_workers=2, pin_memory=DISPOSITIVO == 'cuda')
    return {
        'train': DataLoader(Caras(IDX['train'], transformaciones(nombre, True)), shuffle=True, drop_last=True,
                            generator=g, **kw),
        'val':   DataLoader(Caras(IDX['val'], transformaciones(nombre, False)), **kw) if len(IDX['val']) else None,
        'test':  DataLoader(Caras(IDX['test'], transformaciones(nombre, False)), **kw),
    }


# Ver el aumento de datos
dl = cargadores('C_cnn_cero')['train']
xb, yb = next(iter(dl))
fig, ejes = plt.subplots(2, 8, figsize=(14, 4))
for ax, x, y in zip(ejes.flat, xb, yb):
    ax.imshow((x.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)); ax.set_title(CLASES[y][:12], fontsize=7); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Modelos

- **C · CNN desde cero:** 4 bloques (conv 3×3 → BN → ReLU, dos veces, y *max-pool*), de 32 a 256 filtros
  (112 → 56 → 28 → 14 → 7), *global average pooling*, *dropout* 0.4 y la capa de salida. ≈ 1.2 M parámetros, todos aleatorios.
- **B1 · InceptionResnetV1 (VGGFace2):** red de FaceNet entrenada con 3.3 M fotos de 9 mil personas. Se le pone una
  capa de salida nueva de 29 personas.
- **B2 · ResNet-18 (ImageNet):** preentrenada en objetos, no en caras. Sirve para ver si importa *en qué* se preentrenó.

El *fine-tuning* va en dos fases: primero solo la capa nueva (la red congelada, para no destruir lo aprendido con
gradientes de una capa aleatoria) y luego se descongelan más capas con una tasa de aprendizaje menor.

In [ ]:
def bloque(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))


class CNNCero(nn.Module):
    def __init__(self, n_clases):
        super().__init__()
        self.features = nn.Sequential(bloque(3, 32), bloque(32, 64), bloque(64, 128), bloque(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, n_clases))

    def forward(self, x):
        return self.head(self.features(x))


def crear_modelo(nombre):
    if nombre == 'C_cnn_cero':
        return CNNCero(N_CLASES)
    if nombre == 'B1_facenet_ft':
        from facenet_pytorch import InceptionResnetV1
        # classify=True + num_classes: carga los pesos VGGFace2 y pone una capa 'logits' nueva de N_CLASES
        return InceptionResnetV1(pretrained='vggface2', classify=True, num_classes=N_CLASES)
    if nombre == 'B2_resnet18_ft':
        m = torchvision.models.resnet18(weights='IMAGENET1K_V1')
        m.fc = nn.Linear(m.fc.in_features, N_CLASES)
        return m
    raise ValueError(nombre)


# Fases de entrenamiento: (epocas, que modulos se entrenan, tasa de aprendizaje). 'todo' = toda la red.
FASES = {
    'C_cnn_cero':     [(45, 'todo', 2e-3)],
    'B1_facenet_ft':  [(4, ['logits'], 1e-3),
                       (16, ['repeat_3', 'block8', 'last_linear', 'last_bn', 'logits'], 1e-4)],
    'B2_resnet18_ft': [(4, ['fc'], 1e-3),
                       (20, 'todo', 3e-4)],
}


def contar_parametros(m):
    return sum(p.numel() for p in m.parameters())


for n in MODELOS:
    try:
        print(f'{n:15s} {contar_parametros(crear_modelo(n)) / 1e6:6.2f} M parametros')
    except Exception as e:
        print(f'{n:15s} NO DISPONIBLE: {e}')

## 6. Entrenamiento

AdamW, *label smoothing* 0.1, tasa con coseno en cada fase y precisión mixta (AMP) en GPU. Se guarda el estado con
**mejor accuracy en validación** (si alguna persona no tiene *val*, igual cuenta el resto). Test no se toca aquí.

In [ ]:
def preparar_fase(modelo, modulos):
    # Congela todo menos `modulos`. Devuelve los modulos congelados para dejar su BatchNorm en eval.
    if modulos == 'todo':
        for p in modelo.parameters():
            p.requires_grad = True
        return []
    congelados = []
    for nombre, hijo in modelo.named_children():
        entrena = nombre in modulos
        for p in hijo.parameters():
            p.requires_grad = entrena
        if not entrena:
            congelados.append(hijo)
    return congelados


@torch.no_grad()
def predecir(modelo, dl):
    modelo.eval()
    logits, ys = [], []
    for x, y in dl:
        with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == 'cuda'):
            logits.append(modelo(x.to(DISPOSITIVO, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(logits), torch.cat(ys)


def entrenar(nombre):
    fijar_semilla()
    dls = cargadores(nombre)
    modelo = crear_modelo(nombre).to(DISPOSITIVO)
    criterio = nn.CrossEntropyLoss(label_smoothing=0.1)
    escalador = torch.amp.GradScaler(enabled=DISPOSITIVO == 'cuda')
    historia, mejor = [], dict(acc=-1, estado=None, epoca=0)
    epoca_global, t0 = 0, time.time()

    for n_fase, (epocas, modulos, lr) in enumerate(FASES[nombre], 1):
        congelados = preparar_fase(modelo, modulos)
        params = [p for p in modelo.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=5e-4)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epocas * len(dls['train']),
                                                    pct_start=0.15)
        print(f'--- {nombre} fase {n_fase}: {epocas} epocas, lr {lr}, '
              f'{sum(p.numel() for p in params) / 1e6:.2f} M parametros entrenables')
        for _ in range(epocas):
            epoca_global += 1
            modelo.train()
            for m in congelados:
                m.eval()                      # BatchNorm de lo congelado conserva sus estadisticas
            perdida, aciertos, n = 0.0, 0, 0
            for x, y in dls['train']:
                x, y = x.to(DISPOSITIVO, non_blocking=True), y.to(DISPOSITIVO, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == 'cuda'):
                    salida = modelo(x)
                    loss = criterio(salida, y)
                escalador.scale(loss).backward()
                escalador.step(opt); escalador.update(); sched.step()
                perdida += loss.item() * len(y); aciertos += (salida.argmax(1) == y).sum().item(); n += len(y)
            fila = dict(epoca=epoca_global, fase=n_fase, loss_train=perdida / n, acc_train=aciertos / n)
            if dls['val'] is not None:
                lv, yv = predecir(modelo, dls['val'])
                fila.update(loss_val=F.cross_entropy(lv, yv).item(), acc_val=(lv.argmax(1) == yv).float().mean().item())
                criterio_mejor = fila['acc_val']
            else:
                criterio_mejor = fila['acc_train']
            historia.append(fila)
            if criterio_mejor > mejor['acc']:
                mejor = dict(acc=criterio_mejor, estado=copy.deepcopy(modelo.state_dict()), epoca=epoca_global)
            print(f"ep {epoca_global:3d} | loss {fila['loss_train']:.3f} acc {fila['acc_train']:.3f}"
                  + (f" | val loss {fila['loss_val']:.3f} acc {fila['acc_val']:.3f}" if 'acc_val' in fila else ''))

    modelo.load_state_dict(mejor['estado'])
    info = dict(segundos_entrenamiento=round(time.time() - t0), mejor_epoca=mejor['epoca'],
                parametros=contar_parametros(modelo),
                parametros_entrenables_final=sum(p.numel() for p in modelo.parameters() if p.requires_grad))
    print(f"Mejor epoca {mejor['epoca']} (acc val {mejor['acc']:.3f}) | {info['segundos_entrenamiento']} s")
    return modelo, pd.DataFrame(historia), info, dls

## 7. Evaluación en *test* (una sola vez)

Por modelo se guarda en `SALIDA/<modelo>/`: `metricas.json`, `historia.csv`, `predicciones.csv`, `matriz_confusion.csv`,
`reporte_por_clase.txt` y los pesos `modelo.pt`. Además del accuracy: F1-macro (todas las personas pesan igual),
top-3 y accuracy por **orientación de la cabeza** (frente, perfiles, arriba, abajo…).

In [ ]:
def evaluar(nombre, y_real, y_pred, puntajes=None, historia=None, info=None, modelo=None):
    carpeta = Path(SALIDA) / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    test = datos.iloc[IDX['test']].reset_index(drop=True)
    etiquetas = list(range(N_CLASES))
    met = dict(modelo=nombre, n_test=int(len(y_real)),
               accuracy=accuracy_score(y_real, y_pred),
               f1_macro=f1_score(y_real, y_pred, average='macro', labels=etiquetas, zero_division=0),
               balanced_accuracy=balanced_accuracy_score(y_real, y_pred))
    if puntajes is not None:
        met['top3_accuracy'] = top_k_accuracy_score(y_real, puntajes, k=3, labels=etiquetas)
    if 'orientacion' in test:
        bien = pd.Series(np.asarray(y_real) == np.asarray(y_pred))
        met['accuracy_por_orientacion'] = bien.groupby(test['orientacion']).mean().round(4).to_dict()
    met.update(info or {})

    (carpeta / 'metricas.json').write_text(json.dumps(met, indent=2, ensure_ascii=False, default=float))
    pd.DataFrame(confusion_matrix(y_real, y_pred, labels=etiquetas), index=CLASES, columns=CLASES) \
        .to_csv(carpeta / 'matriz_confusion.csv')
    (carpeta / 'reporte_por_clase.txt').write_text(
        classification_report(y_real, y_pred, labels=etiquetas, target_names=CLASES, digits=3, zero_division=0))
    pd.DataFrame(dict(ruta=test['ruta'], set=test['set'], real=[CLASES[i] for i in y_real],
                      predicha=[CLASES[i] for i in y_pred])).to_csv(carpeta / 'predicciones.csv', index=False)
    if historia is not None:
        historia.to_csv(carpeta / 'historia.csv', index=False)
    if modelo is not None:
        torch.save(dict(estado=modelo.state_dict(), clases=CLASES, normalizacion=NORMAS.get(nombre)), carpeta / 'modelo.pt')
    print(f"{nombre}: accuracy {met['accuracy']:.4f} | F1-macro {met['f1_macro']:.4f}"
          + (f" | top-3 {met['top3_accuracy']:.4f}" if 'top3_accuracy' in met else ''))
    return met


RESULTADOS, HISTORIAS, PREDICCIONES = {}, {}, {}
ENTRENADOS = {}      # nombre -> modelo de torch (en CPU) o clasificador de sklearn, para la seccion 9

### 7.1 Línea base A: ArcFace congelado + Ridge / SVM / KNN (el enfoque de Kenya)

Se calculan aquí los vectores de 512 números de **ArcFace antelopev2** (`glintr100.onnx` de InsightFace, ~360 MB, uso no
comercial) para cada cara y se entrenan los mismos tres clasificadores que usa Kenya, **solo con *train***.
Diferencia con su pipeline: allá las caras se alinean con los puntos de los ojos antes de ArcFace; aquí se usan los
recortes de MediaPipe reescalados a 112×112, así que este A puede quedar un poco por debajo del suyo.

In [ ]:
# --- ArcFace antelopev2 (se usa en A y en la prueba de robustez) ---
import zipfile
ARCFACE_SESION = None


def sesion_arcface():
    # ArcFace antelopev2 (glintr100.onnx) de InsightFace, igual que src/comun.py de Kenya.
    global ARCFACE_SESION
    if ARCFACE_SESION is None:
        import subprocess, sys, urllib.request
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'], check=False)
        import onnxruntime as ort
        destino = Path('/content/model_antelope.onnx')
        if not destino.exists():
            zpath = Path('/content/antelopev2.zip')
            print('Descargando antelopev2 (~360 MB) ...')
            urllib.request.urlretrieve('https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip', zpath)
            with zipfile.ZipFile(zpath) as z:
                miembro = next(n for n in z.namelist() if n.endswith('glintr100.onnx'))
                with z.open(miembro) as o, open(destino, 'wb') as d:
                    shutil.copyfileobj(o, d)
        ort.set_default_logger_severity(3)
        ARCFACE_SESION = ort.InferenceSession(str(destino), providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        print('ArcFace en', ARCFACE_SESION.get_providers()[0])
    return ARCFACE_SESION


def vectores_arcface(X, lote=64):
    from sklearn.preprocessing import normalize
    s = sesion_arcface(); entrada = s.get_inputs()[0].name
    salida = []
    for i in range(0, len(X), lote):
        x = ((X[i:i + lote].astype(np.float32) - 127.5) / 127.5).transpose(0, 3, 1, 2)
        salida.append(s.run(None, {entrada: np.ascontiguousarray(x)})[0])
    return normalize(np.vstack(salida), norm='l2')


if CORRER_LINEA_BASE_A:
    from sklearn.linear_model import RidgeClassifier
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier

    t0 = time.time()
    V = vectores_arcface(X_TODO)                  # mismo orden que `datos`
    print(f'Vectores ArcFace: {V.shape} en {time.time() - t0:.0f} s')
    clasificadores = {   # mismos hiperparametros que src/exportar_modelos.py de Kenya
        'A_arcface_ridge': RidgeClassifier(alpha=1.0),
        'A_arcface_svm':   SVC(kernel='linear', C=1.0),
        'A_arcface_knn':   KNeighborsClassifier(n_neighbors=5, metric='cosine', algorithm='brute'),
    }
    for nombre, clf in clasificadores.items():
        t0 = time.time()
        clf.fit(V[IDX['train']], Y_TODO[IDX['train']])
        segundos = round(time.time() - t0, 1)
        y_pred = clf.predict(V[IDX['test']])
        puntajes = clf.decision_function(V[IDX['test']]) if hasattr(clf, 'decision_function') \
            else clf.predict_proba(V[IDX['test']])
        RESULTADOS[nombre] = evaluar(nombre, Y_TODO[IDX['test']], y_pred, puntajes,
                                     info=dict(segundos_entrenamiento=segundos, parametros_entrenables_final=None))
        PREDICCIONES[nombre] = y_pred
        ENTRENADOS[nombre] = clf
else:
    print('Linea base A omitida (CORRER_LINEA_BASE_A = False).')

### 7.2 Modelos B y C

In [ ]:
for nombre in MODELOS:
    try:
        modelo, historia, info, dls = entrenar(nombre)
    except ImportError as e:
        print(f'{nombre} omitido: {e}')
        continue
    logits, y_test = predecir(modelo, dls['test'])
    y_pred = logits.argmax(1).numpy()
    RESULTADOS[nombre] = evaluar(nombre, y_test.numpy(), y_pred, logits.softmax(1).numpy(), historia, info, modelo)
    HISTORIAS[nombre], PREDICCIONES[nombre] = historia, y_pred
    ENTRENADOS[nombre] = modelo.cpu().eval()
    del modelo; torch.cuda.empty_cache()

## 8. Comparación

In [ ]:
cols = ['accuracy', 'f1_macro', 'balanced_accuracy', 'top3_accuracy', 'segundos_entrenamiento', 'parametros_entrenables_final']
comparacion = pd.DataFrame(RESULTADOS).T.reindex(columns=cols).astype(float)
comparacion = comparacion.sort_values('f1_macro', ascending=False)
comparacion.to_csv(Path(SALIDA) / 'comparacion.csv')
display(comparacion.style.format({c: '{:.4f}' for c in cols[:4]}, na_rep='—'))

orient = pd.DataFrame({k: v.get('accuracy_por_orientacion', {}) for k, v in RESULTADOS.items()})
if not orient.empty:
    print('\nAccuracy en test por orientacion de la cabeza:')
    display(orient.style.format('{:.3f}', na_rep='—').background_gradient(axis=None, cmap='RdYlGn'))
    orient.to_csv(Path(SALIDA) / 'accuracy_por_orientacion.csv')

fig, ax = plt.subplots(figsize=(8, 3.5))
comparacion[['accuracy', 'f1_macro']].plot.barh(ax=ax)
ax.set_xlim(0, 1); ax.set_title('Test (sets no vistos)'); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(Path(SALIDA) / 'comparacion.png', dpi=150); plt.show()

In [ ]:
# Curvas de aprendizaje
if HISTORIAS:
    fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
    for nombre, h in HISTORIAS.items():
        l = ejes[0].plot(h['epoca'], h['loss_train'], '--', alpha=0.6)[0]
        if 'loss_val' in h:
            ejes[0].plot(h['epoca'], h['loss_val'], color=l.get_color(), label=nombre)
        ejes[1].plot(h['epoca'], h['acc_train'], '--', color=l.get_color(), alpha=0.6)
        if 'acc_val' in h:
            ejes[1].plot(h['epoca'], h['acc_val'], color=l.get_color(), label=nombre)
    ejes[0].set_title('Perdida (punteado = train, continuo = val)'); ejes[1].set_title('Accuracy')
    for ax in ejes:
        ax.set_xlabel('epoca'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(Path(SALIDA) / 'curvas.png', dpi=150); plt.show()

## 9. Prueba difícil: robustez y fotos nuevas

En *test* (sets no vistos) todos rondan el 100 %, así que se repite la prueba con las **mismas caras de test
modificadas** para simular el disfraz del examen y condiciones difíciles:

| Condición | Cómo se simula sobre la cara alineada 112×112 |
|---|---|
| `lentes_oscuros` | franja negra sobre los ojos |
| `gorra` | franja oscura sobre la frente y el cabello |
| `cubrebocas` | mitad inferior tapada (color claro) |
| `borrosa` | desenfoque gaussiano (σ = 2.5) |
| `poca_luz` | brillo al 30 % + ruido |
| `baja_resolucion` | se reduce a 20×20 px y se vuelve a ampliar |

Se usa una submuestra fija de test (hasta 40 caras por persona), igual para todos los modelos. Para el modelo A se
recalculan los vectores ArcFace de cada cara modificada (se descarga `antelopev2` de InsightFace, ~360 MB, uso no comercial).

Además, si el zip trae una carpeta **`pruebas_unicas_arcface/<persona>/*.png`** (fotos nuevas fuera del dataset,
alineadas con `python src/alinear.py --entrada data/PRUEBAS_UNICAS --salida data/pruebas_unicas_arcface`), se evalúan
todos los modelos con ellas. Nunca se usan para entrenar.

In [ ]:
from PIL import ImageFilter

rng_pert = np.random.default_rng(SEMILLA)


def lentes_oscuros(c):
    c = c.copy(); c[38:62, 18:94] = 15; return c


def gorra(c):
    c = c.copy(); c[0:34, :] = (38, 40, 46); return c


def cubrebocas(c):
    c = c.copy(); c[68:112, 8:104] = (205, 215, 228); return c


def borrosa(c):
    return np.asarray(Image.fromarray(c).filter(ImageFilter.GaussianBlur(2.5)))


def poca_luz(c):
    x = c.astype(np.float32) * 0.30 + rng_pert.normal(0, 6, c.shape)
    return np.clip(x, 0, 255).astype(np.uint8)


def baja_resolucion(c):
    return np.asarray(Image.fromarray(c).resize((20, 20), Image.BILINEAR).resize((112, 112), Image.BILINEAR))


PERTURBACIONES = {'original': lambda c: c, 'lentes_oscuros': lentes_oscuros, 'gorra': gorra, 'cubrebocas': cubrebocas,
                  'borrosa': borrosa, 'poca_luz': poca_luz, 'baja_resolucion': baja_resolucion}

# Submuestra fija de test: hasta 40 caras por persona (posiciones dentro de `datos`)
test_df = datos.iloc[IDX['test']]
sub = np.concatenate([g.sample(min(len(g), 40), random_state=SEMILLA).index.to_numpy()
                      for _, g in test_df.groupby('persona')])
X_SUB, Y_SUB = X_TODO[sub], Y_TODO[sub]
X_PERT = {k: np.stack([f(c) for c in X_SUB]) for k, f in PERTURBACIONES.items()}
print(f'{len(sub)} caras de test por condicion')

fig, ejes = plt.subplots(1, len(X_PERT), figsize=(2.2 * len(X_PERT), 2.6))
for ax, (k, X) in zip(ejes, X_PERT.items()):
    ax.imshow(X[0]); ax.set_title(k, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(Path(SALIDA) / 'perturbaciones_ejemplo.png', dpi=120); plt.show()

In [ ]:
# --- Prediccion sobre arreglos de caras (N, 112, 112, 3) uint8 para cualquier modelo ---------------------------
@torch.no_grad()
def predecir_arreglo(nombre, X, vectores=None):
    m = ENTRENADOS[nombre]
    if nombre.startswith('A_'):
        return m.predict(vectores if vectores is not None else vectores_arcface(X))
    tf = transformaciones(nombre, entrenar=False)
    m = m.to(DISPOSITIVO).eval(); pred = []
    for i in range(0, len(X), 128):
        x = torch.stack([tf(torch.from_numpy(c).permute(2, 0, 1)) for c in X[i:i + 128]]).to(DISPOSITIVO)
        with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == 'cuda'):
            pred.append(m(x).argmax(1).cpu())
    ENTRENADOS[nombre] = m.cpu()
    return torch.cat(pred).numpy()


# --- Robustez -------------------------------------------------------------------------------------------------
hay_A = any(n.startswith('A_') for n in ENTRENADOS)
VEC_PERT = {k: vectores_arcface(X) for k, X in X_PERT.items()} if hay_A else {}
if hay_A:
    # chequeo: los vectores recalculados de las caras originales deben coincidir con los de la sección 7.1
    sim = (VEC_PERT['original'] * V[sub]).sum(1)
    print(f'Similitud con los vectores de 7.1 (deberia ser ~1): media {sim.mean():.4f}, minima {sim.min():.4f}')

robustez = {}
for nombre in ENTRENADOS:
    robustez[nombre] = {k: accuracy_score(Y_SUB, predecir_arreglo(nombre, X, VEC_PERT.get(k)))
                        for k, X in X_PERT.items()}
robustez = pd.DataFrame(robustez).T
robustez['promedio_dificiles'] = robustez.drop(columns='original').mean(1)
robustez = robustez.sort_values('promedio_dificiles', ascending=False)
robustez.to_csv(Path(SALIDA) / 'robustez.csv')
display(robustez.style.format('{:.3f}').background_gradient(axis=None, cmap='RdYlGn', vmin=0, vmax=1))

ax = robustez.drop(columns='promedio_dificiles').T.plot.bar(figsize=(11, 4), width=0.8)
ax.set_ylim(0, 1); ax.set_ylabel('accuracy'); ax.set_title('Robustez en test (misma submuestra para todos)')
ax.legend(fontsize=8, ncol=3); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(Path(SALIDA) / 'robustez.png', dpi=150); plt.show()

In [ ]:
# --- Fotos nuevas fuera del dataset (opcional) -----------------------------------------------------------------
PU_DIR = next((p for p in TRABAJO.rglob('pruebas_unicas_arcface') if p.is_dir()), None)
if PU_DIR is None:
    print('Sin pruebas_unicas_arcface/: se omite (ver seccion 9).')
else:
    rutas = sorted(PU_DIR.rglob('*.png'))
    personas = [r.relative_to(PU_DIR).parts[0] for r in rutas]
    validas = [i for i, p in enumerate(personas) if p in A_INDICE]
    ignoradas = sorted(set(personas) - set(A_INDICE))
    if ignoradas:
        print('Carpetas que no son ninguna de las personas entrenadas (se ignoran):', ignoradas)
    rutas = [rutas[i] for i in validas]
    X_PU = cargar(rutas); Y_PU = np.array([A_INDICE[personas[i]] for i in validas])
    V_PU = vectores_arcface(X_PU) if hay_A else None
    filas = []
    for nombre in ENTRENADOS:
        pred = predecir_arreglo(nombre, X_PU, V_PU if nombre.startswith('A_') else None)
        filas.append(dict(modelo=nombre, n=len(Y_PU), accuracy=accuracy_score(Y_PU, pred)))
        pd.DataFrame(dict(foto=[str(r.relative_to(PU_DIR)) for r in rutas], real=[CLASES[i] for i in Y_PU],
                          predicha=[CLASES[i] for i in pred])).to_csv(Path(SALIDA) / nombre / 'pruebas_unicas_predicciones.csv', index=False)
    pu = pd.DataFrame(filas).set_index('modelo').sort_values('accuracy', ascending=False)
    pu.to_csv(Path(SALIDA) / 'pruebas_unicas.csv')
    display(pu)

## 10. Análisis de errores del mejor modelo

In [ ]:
mejor = comparacion.index[0]
cm = pd.read_csv(Path(SALIDA) / mejor / 'matriz_confusion.csv', index_col=0)
fig, ax = plt.subplots(figsize=(10, 9))
cm_norm = cm.div(cm.sum(1).replace(0, 1), axis=0)
ax.imshow(cm_norm, cmap='Purples')
ax.set_xticks(range(N_CLASES)); ax.set_xticklabels([c[:12] for c in CLASES], rotation=90, fontsize=7)
ax.set_yticks(range(N_CLASES)); ax.set_yticklabels([c[:12] for c in CLASES], fontsize=7)
ax.set_xlabel('predicha'); ax.set_ylabel('real'); ax.set_title(f'Matriz de confusion normalizada · {mejor}')
plt.tight_layout(); plt.savefig(Path(SALIDA) / mejor / 'matriz_confusion.png', dpi=150); plt.show()

# Pares que mas se confunden
pares = [(CLASES[i], CLASES[j], int(cm.iat[i, j])) for i in range(N_CLASES) for j in range(N_CLASES) if i != j and cm.iat[i, j]]
display(pd.DataFrame(sorted(pares, key=lambda t: -t[2])[:10], columns=['real', 'predicha', 'veces']))

# Por persona: accuracy en test
print((pd.Series(np.diag(cm_norm), index=CLASES).sort_values()).round(3).head(10).to_string())

# Galeria de errores
y_test = Y_TODO[IDX['test']]
errores = np.flatnonzero(PREDICCIONES[mejor] != y_test)[:24]
if len(errores):
    fig, ejes = plt.subplots(3, 8, figsize=(15, 6.5))
    for ax in ejes.flat:
        ax.axis('off')
    for ax, k in zip(ejes.flat, errores):
        ax.imshow(X_TODO[IDX['test'][k]])
        ax.set_title(f'{CLASES[y_test[k]][:10]}\n→ {CLASES[PREDICCIONES[mejor][k]][:10]}', fontsize=7, color='crimson')
    plt.suptitle(f'Errores en test · {mejor} (real → predicha)'); plt.tight_layout(); plt.show()

## 11. Qué poner en el reporte

- Tabla de `comparacion.csv` + `comparacion.png` y `curvas.png`.
- **A vs B1:** ¿entrenar la red ayuda, o los vectores de ArcFace ya bastan?
- **B1 vs B2:** preentrenar en caras contra preentrenar en objetos.
- **B vs C:** qué tanto aporta el preentrenamiento con ~300 fotos por persona. Si C se sobreajusta, se ve en `curvas.png`
  (train sube y val se estanca).
- `accuracy_por_orientacion.csv`: en qué ángulos falla cada modelo.
- `robustez.csv` / `robustez.png`: **la comparación que sí separa a los modelos** (lentes, cubrebocas, blur…).
- `pruebas_unicas.csv`, si hubo fotos nuevas: la prueba "de la vida real".
- Limitación: siempre responde una de las 29 personas (no hay "desconocido").
- **División:** con `por_set` la prueba usa sesiones que el modelo nunca vio; es la que se parece al examen. Con
  `aleatoria` los resultados suben porque se mezclan fotos de la misma sesión.

## 12. Descargar resultados

In [ ]:
!cd "{Path(SALIDA).parent}" && zip -qr /content/{Path(SALIDA).name}.zip {Path(SALIDA).name} -x "*.pt"
from google.colab import files
files.download(f'/content/{Path(SALIDA).name}.zip')   # sin pesos (pesan mucho); los pesos se quedan en el Drive